# 结业项目三预置笔记本：写最小 eval

题面：你的 AI 客服总是忘了附订单号。写个最小 eval，量化它到底多经常忘。

最小 eval 五件：任务定义一句话｜条目级数据集（失败用例先行）｜判定器（优先规则）｜条目级报表｜基线与门槛。本笔记本与同目录 eval_min.py 是同一份代码的两个入口，任何一处改动两边同步。

## 1. 任务定义与数据集

任务定义一句话：客服回复必须包含用户消息里的订单号；用户没给订单号则不强制。

24 条用例分三区：失败用例区 F01–F10（先于任何修复写成）、常规用例区 P01–P10、边界用例区 E01–E04。门槛：24 条至少 22 条（≈91.7%）算修好。

In [1]:
from eval_min import CASES, THRESHOLD, extract_order_no
print(f'用例总数 {len(CASES)}，门槛 {THRESHOLD}/{len(CASES)}')
from collections import Counter
print(Counter(c['zone'] for c in CASES))

用例总数 24，门槛 22/24
Counter({'失败用例': 10, '常规用例': 10, '边界用例': 4})


## 2. 判定器自检

规则判定：抽出订单号，检查回复是否包含它。判定器自己也要先验货，这步叫判定器自检。

In [2]:
for c in CASES:
    print(c['id'], '->', extract_order_no(c['user']))

F01 -> #A1024
F02 -> B-2048
F03 -> C7788
F04 -> #D3092
F05 -> E-5510
F06 -> F6600
F07 -> G-8817
F08 -> H9902
F09 -> #I1230
F10 -> J-4476
P01 -> #A1024
P02 -> B-2048
P03 -> C7788
P04 -> #D3092
P05 -> E-5510
P06 -> F6600
P07 -> G-8817
P08 -> H9902
P09 -> #I1230
P10 -> J-4476
E01 -> #K5521
E02 -> L8820
E03 -> None
E04 -> m-3355


## 3. 基线：规则小模型 v1（修复前）

In [3]:
from eval_min import rule_bot_v1, run_eval
rows_v1 = run_eval('规则小模型 v1（修复前）', rule_bot_v1, CASES)


===== 规则小模型 v1（修复前） =====
耗时 0.0 s
F01 [失败用例] FAIL  订单号 #A1024 未复述
    用户: 你好，我上周下的单一直没到，订单号 #A1024，能帮我问问物流吗？物流单号我这边查不到，麻烦尽快。
    回复: 您好，您的问题我们已经记录，会尽快为您核实处理，请保持电话畅通。
F02 [失败用例] FAIL  订单号 B-2048 未复述
    用户: 订单号 B-2048。我要改地址，原地址写错了，新地址是杭州市西湖区文三路 100 号，收件人电话不变，请帮我更新，谢谢！
    回复: 您好，您的问题我们已经记录，会尽快为您核实处理，请保持电话畅通。
F03 [失败用例] FAIL  订单号 C7788 未复述
    用户: 亲，我买了一件外套想退货，订单号 C7788，衣服尺码偏小穿不了，吊牌还在，包装完整，麻烦帮我处理一下退款哦。
    回复: 您好，您的问题我们已经记录，会尽快为您核实处理，请保持电话畅通。
F04 [失败用例] FAIL  订单号 #D3092 未复述
    用户: 你好，订单号 #D3092，我收到的杯子碎了一个角，外包装是完好的，应该是运输途中磕的，麻烦补发一个或者给我退款，谢谢。
    回复: 您好，您的问题我们已经记录，会尽快为您核实处理，请保持电话畅通。
F05 [失败用例] FAIL  订单号 E-5510 未复述
    用户: 我的订单号 E-5510，买了三本书只到了两本，缺的那本是《时间简史》，快递面单上写的也是两本，请帮我核实一下仓库发货记录。
    回复: 您好，您的问题我们已经记录，会尽快为您核实处理，请保持电话畅通。
F06 [失败用例] FAIL  订单号 F6600 未复述
    用户: 订单号：F6600。请问我的发票什么时候能开？公司报销要用，抬头是'杭州某某科技有限公司'，税号 91330106MA27xxxx33，谢谢！
    回复: 您好，您的问题我们已经记录，会尽快为您核实处理，请保持电话畅通。
F07 [失败用例] FAIL  订单号 G-8817 未复述
    用户: 你好呀，我想问一下订单号 G-8817 的包裹到哪儿了？显示签收但我没收到，快递员电话也打不通，物业那里也找过了，麻烦帮我查查。
    回复: 您好

## 4. eval-driven：失败用例先于改动存在，改动由失败用例驱动

看 v1 的分区报表：失败用例区全挂，就是改动的清单。修完用同一批用例裁定。

In [4]:
from eval_min import rule_bot_v2, run_eval
rows_v2 = run_eval('规则小模型 v2（修复后）', rule_bot_v2, CASES)


===== 规则小模型 v2（修复后） =====
耗时 0.0 s
F01 [失败用例] PASS  订单号 #A1024 已复述
    用户: 你好，我上周下的单一直没到，订单号 #A1024，能帮我问问物流吗？物流单号我这边查不到，麻烦尽快。
    回复: 您好，已收到您关于订单 #A1024 的问题，我们马上为您处理，请保持电话畅通。
F02 [失败用例] PASS  订单号 B-2048 已复述
    用户: 订单号 B-2048。我要改地址，原地址写错了，新地址是杭州市西湖区文三路 100 号，收件人电话不变，请帮我更新，谢谢！
    回复: 您好，已收到您关于订单 B-2048 的问题，我们马上为您处理，请保持电话畅通。
F03 [失败用例] PASS  订单号 C7788 已复述
    用户: 亲，我买了一件外套想退货，订单号 C7788，衣服尺码偏小穿不了，吊牌还在，包装完整，麻烦帮我处理一下退款哦。
    回复: 您好，已收到您关于订单 C7788 的问题，我们马上为您处理，请保持电话畅通。
F04 [失败用例] PASS  订单号 #D3092 已复述
    用户: 你好，订单号 #D3092，我收到的杯子碎了一个角，外包装是完好的，应该是运输途中磕的，麻烦补发一个或者给我退款，谢谢。
    回复: 您好，已收到您关于订单 #D3092 的问题，我们马上为您处理，请保持电话畅通。
F05 [失败用例] PASS  订单号 E-5510 已复述
    用户: 我的订单号 E-5510，买了三本书只到了两本，缺的那本是《时间简史》，快递面单上写的也是两本，请帮我核实一下仓库发货记录。
    回复: 您好，已收到您关于订单 E-5510 的问题，我们马上为您处理，请保持电话畅通。
F06 [失败用例] PASS  订单号 F6600 已复述
    用户: 订单号：F6600。请问我的发票什么时候能开？公司报销要用，抬头是'杭州某某科技有限公司'，税号 91330106MA27xxxx33，谢谢！
    回复: 您好，已收到您关于订单 F6600 的问题，我们马上为您处理，请保持电话畅通。
F07 [失败用例] PASS  订单号 G-8817 已复述
    用户: 你好呀，我想问一下订单号 G-8817 的包裹到哪儿了？显

## 5. 被评对象换成真实模型：MiniMind2（本机 CPU，可选）

不提醒它附订单号，贪心解码可复现。没有 modelscope 环境时本格会自动跳过。

In [5]:
try:
    from eval_min import load_minimind, minimind_reply, run_eval
    tok, model = load_minimind()
    rows_llm = run_eval('MiniMind2（未提醒附订单号）',
                        lambda u: minimind_reply(tok, model, u), CASES)
except ModuleNotFoundError as e:
    print('跳过：本环境缺依赖', e)

<site-packages>\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-10-09 23:52:46,936 | INFO    | modelscope_hub.download | Downloading 29 files from gongjy/minimind2@master


Downloading:   0%|          | 0/29 [00:00<?, ?file/s]

Downloading: 100%|██████████| 29/29 [00:00<00:00, 169.38file/s]

Downloading: 100%|██████████| 29/29 [00:00<00:00, 168.40file/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 146/146 [00:00<00:00, 36483.07it/s]


===== MiniMind2（未提醒附订单号） =====
耗时 17.1 s
F01 [失败用例] PASS  订单号 #A1024 已复述
    用户: 你好，我上周下的单一直没到，订单号 #A1024，能帮我问问物流吗？物流单号我这边查不到，麻烦尽快。
    回复: 你好！很抱歉听到你上周下的单一直没到，订单号 #A1024，这可能意味着你的订单信息中包含了一些不明确或不完整的信息。为了帮助你
F02 [失败用例] FAIL  订单号 B-2048 未复述
    用户: 订单号 B-2048。我要改地址，原地址写错了，新地址是杭州市西湖区文三路 100 号，收件人电话不变，请帮我更新，谢谢！
    回复: 当然，我很乐意帮助您更新订单号。请将您的订单号更新到以下位置：

1. 杭州市西湖区文三路 100 号
2.
F03 [失败用例] PASS  订单号 C7788 已复述
    用户: 亲，我买了一件外套想退货，订单号 C7788，衣服尺码偏小穿不了，吊牌还在，包装完整，麻烦帮我处理一下退款哦。
    回复: 亲，您好，我是电商客服，很高兴为您服务。请问您购买的外套是否正常？订单号 C7788 是否已经退货？衣服尺码偏小
F04 [失败用例] PASS  订单号 #D3092 已复述
    用户: 你好，订单号 #D3092，我收到的杯子碎了一个角，外包装是完好的，应该是运输途中磕的，麻烦补发一个或者给我退款，谢谢。
    回复: 你好，订单号 #D3092，我收到的是一条运输途中磕的角，外包装是完好的，我需要退款。
F05 [失败用例] PASS  订单号 E-5510 已复述
    用户: 我的订单号 E-5510，买了三本书只到了两本，缺的那本是《时间简史》，快递面单上写的也是两本，请帮我核实一下仓库发货记录。
    回复: 根据您提供的信息，您似乎在询问关于订单号 E-5510 的详细信息，以及您在购买过程中遇到的具体问题。为了提供更准确的帮助，我需要更多的上下
F06 [失败用例] PASS  订单号 F6600 已复述
    用户: 订单号：F6600。请问我的发票什么时候能开？公司报销要用，抬头是'杭州某某科技有限公司'，税号 91330106MA27xxxx33，谢谢！
    回复: 根据您提供的信息，您的订单号为

## 6. 进阶：把判定器换成 LLM 评判员，亲身体会三偏差

judge_swap.py 用 MiniMind2 扮演评判员做换序测试。2026-10-09 本机实测：0.1B 评判员不守指令、多数答错、结论随顺序漂移，整体不合格。这本身就是评判员也要被评估的证据。

In [6]:
import subprocess, sys
r = subprocess.run([sys.executable, 'judge_swap.py'],
                   capture_output=True, text=True, encoding='utf-8')
print(r.stdout[-1500:])

样例1（回复确实复述了订单号，正确答案：是）
  正序（用户在前） -> 评判员回答：'客服回复中包含了用户'
  换序（回复在前） -> 评判员回答：'客服回复：您好，'
样例2（回复没有复述订单号，正确答案：否）
  正序（用户在前） -> 评判员回答：'客服回复中包含了用户'
  换序（回复在前） -> 评判员回答：'是，客服回复：您'
样例3（回复没有复述订单号，正确答案：否）
  正序（用户在前） -> 评判员回答：'是的，您需要查询物'
  换序（回复在前） -> 评判员回答：'是的，您需要查询物'
对照：规则判定器对三对样例的答案是 是／否／否（不随顺序变）。

